In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['MSC_CONFIG'] = "/mnt/nas/home/genggui/code/Megatron-Next/.msc_config.yaml"

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_1_35b_a3b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_1_35b_a3b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config, watchdog_timeout_seconds=240)

rank=0, worker=0: sample_range=[0, 98994] in 1 slices, sum(count)=98994: indexes=[0, 98994] slices=[train-chunk-0.tar[0(start), 98994(end)]]
rank=0, worker=0: sample_range=[0, 33580] in 1 slices, sum(count)=33580: indexes=[0, 33580] slices=[train-chunk-0.tar[0(start), 33580(end)]]
rank=0, worker=0: sample_range=[0, 70519] in 1 slices, sum(count)=70519: indexes=[0, 70519] slices=[train-chunk-0.tar[0(start), 70519(end)]]
rank=0, worker=0: sample_range=[0, 99256] in 1 slices, sum(count)=99256: indexes=[0, 99256] slices=[train-chunk-0.tar[0(start), 99256(end)]]
rank=0, worker=0: sample_range=[0, 14160] in 1 slices, sum(count)=14160: indexes=[0, 14160] slices=[train-chunk-0.tar[0(start), 14160(end)]]
rank=0, worker=0: sample_range=[0, 5231] in 1 slices, sum(count)=5231: indexes=[0, 5231] slices=[train-chunk-0.tar[0(start), 5231(end)]]
rank=0, worker=0: sample_range=[0, 1453] in 1 slices, sum(count)=1453: indexes=[0, 1453] slices=[train-chunk-0.tar[0(start), 1453(end)]]
rank=0, worker=0: sam

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [ ]:
dd = iter(train_dataloader)

/mnt/nas/home/genggui/miniconda3/envs/megatron-next-torch-29/lib/python3.12/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Token indices sequence length is longer than the specified maximum sequence length for this model (94293 > 66536). Running this sequence through the model will result in indexing errors


In [6]:
import json

count_map = {}

with open("/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v20_1_35b_a3b_next_gemini_bf16/qua/pulse_v20_1_35b_a3b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

100%|██████████| 2048/2048 [00:44<00:00, 46.47it/s] 


In [7]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

[('evol-codealpaca-v1', 0.03466796875),
 ('exam-math-cn', 0.025390625),
 ('gg_zh_gg_zh_edu_data', 0.0244140625),
 ('gg_en_fineweb-edu_data', 0.0234375),
 ('预问诊', 0.0224609375),
 ('difficult_med_choice_cn-same-language-think', 0.02197265625),
 ('aya_dataset', 0.02099609375),
 ('function_call', 0.02099609375),
 ('WildChat', 0.0205078125),
 ('diffcult-code-agent-notools-en-same-language-think', 0.02001953125),
 ('zh_med_our_med_kb', 0.01904296875),
 ('exam-en', 0.0185546875),
 ('difficult_PromptCBLUE-same-language-think', 0.017578125),
 ('diffcult-math-en-same-language-think', 0.0166015625),
 ('general-cn-my', 0.0166015625),
 ('multi_my_function_tools', 0.0166015625),
 ('tulu-3-IF-augmented', 0.01513671875),
 ('difficult_med_choice_en-same-language-think', 0.0146484375),
 ('all_agent-english_reasoning_with_interleaved_thinking_use_last_message',
  0.0146484375),
 ('longalign-en-think', 0.01416015625),
 ('gg_zh_med_ns_data', 0.01416015625),
 ('difficult_med_q_cn-same-language-think', 0.013

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [9]:
tokenizer("<|im_start|>assistant\n<ggthink>\n\n</ggthink>\n\n").input_ids 

[248045, 74455, 198, 248068, 271, 248069, 271]

In [10]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids

[248045, 74455, 198, 248068, 198]

In [11]:
tokenizer('<|im_end|>').input_ids

[248046]

In [12]:
for _ in tqdm(range(2048)):
    item = next(dd)

100%|██████████| 2048/2048 [00:48<00:00, 41.95it/s] 


In [26]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>user
数列$ \{a_{n}\} $满足$ a_{1}=5 $，且$ \dfrac {1}{a_{1}}+ \dfrac {1}{a_{2}}+ \dfrac {1}{a_{3}}+…+ \dfrac {1}{a_{n-1}}= \dfrac {2}{a_{n}}(n\geqslant 2,n∈N^{\*}). $   
$ (1) $求$ a_{2} $，$ a_{3} $，$ a_{4} $；   
$ (2) $求数列$ \{a_{n}\} $的通项公式；   
$ (3) $令$ b_{n}= \dfrac {a_{n}}{11-2a_{n}} $，求数列$ \{b_{n}\} $的最大值与最小值．<|im_end|>
<|im_start|>assistant
<ggthink>

</ggthink>

解：
(1) 设 $ S_{n-1} = \dfrac {1}{a_{1}}+ \dfrac {1}{a_{2}}+ \dfrac {1}{a_{3}}+…+ \dfrac {1}{a_{n-1}} $.
由题意得 $ S_{n-1} = \dfrac{2}{a_n} $ 对 $ n \geqslant 2 $ 成立.

当 $ n=2 $ 时，$ S_1 = \dfrac{1}{a_1} = \dfrac{2}{a_2} $.
因为 $ a_1 = 5 $，所以 $ \dfrac{1}{5} = \dfrac{2}{a_2} $，解得 $ a_2 = 10 $.

当 $ n=3 $ 时，$ S_2 = \dfrac{1}{a_1} + \dfrac{1}{a_2} = \dfrac{2}{a_3} $.
将 $ a_1 = 5 $ 和 $ a_2 = 10 $ 代入，得 $ \dfrac{1}{5} + \dfrac{1}{10} = \dfrac{2}{a_3} $.
$ \dfrac{2+1}{10} = \dfrac{3}{10} = \dfrac{2}{a_3} $.
$ 3a_3 = 20 $，解得 $ a_3 = \dfrac{20}{3} $.

当 $ n=4 $ 时，$ S_3 = \dfrac{1}{a_1} + \dfrac{1}{a_2} + \dfrac{1}{a_3} = \dfrac{2}{a

In [14]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

营卫不和导致的汗证，在中医里通常指的是“自汗”，即在没有明显诱因（如剧烈运动、高温环境、服用发汗药物等）的情况下，白天时时汗出，活动后更甚。其核心病机是卫气不能固护肌表，营阴不能内守而外泄。

**主要表现：**
*   时时汗出，动辄益甚。
*   常伴有恶风（怕风）。
*   面色㿠白或少华。
*   体倦乏力。
*   舌淡苔薄白。
*   脉象多浮缓或细弱。

**治疗原则：**
调和营卫，益气固表。

**常用方剂：**
1.  **桂枝汤（《伤寒论》）：**
    *   **组成：** 桂枝、芍药、生姜、大枣、炙甘草。
    *   **功用：** 解肌发表，调和营卫。
    *   **主治：** 这是调和营卫的代表方。方中桂枝辛温，助卫阳以解肌发表；芍药酸寒，益营阴而敛汗。生姜助桂枝发散，大枣助芍药益阴。甘草调和诸药。此方通过发散和收敛的巧妙配合，使卫气和，营阴守，汗自止。
    *   **服用方法及注意事项：** 原方要求服药后喝热稀粥，并盖被取微汗。汗出后，病邪随汗而解，营卫调和，汗证自然停止。如果服药后汗不出，可以再服。如果汗出不止，则说明营卫失调未愈，需要调整。

2.  **玉屏风散（《丹溪心法》）：**
    *   **组成：** 黄芪、白术、防风。
    *   **功用：** 益气固表止汗。
    *   **主治：** 主要用于表虚自汗，恶风，面色㿠白，舌淡苔薄白，脉浮虚。黄芪益气固表为君，白术健脾益气助黄芪固表，防风走表而散风邪，又能助黄芪固表之功。此方更侧重于卫气虚弱，固摄无权导致的自汗。

**如何选择？**
*   **桂枝汤证：** 除了自汗、恶风外，可能还伴有轻微发热、头痛、鼻鸣干呕等太阳中风表虚证的表现。其核心是营卫之间失去协调，卫强营弱或卫弱营强。
*   **玉屏风散证：** 更多表现为单纯的卫气虚弱，肌表不固，汗易外泄，平时易感冒，体质较虚。

**日常调理建议：**
1.  **避风寒：** 避免长时间吹风，尤其是在出汗后。
2.  **适当锻炼：** 进行适度的体育锻炼，如散步、太极拳等，以增强体质，但避免剧烈运动导致大汗淋漓。
3.  **饮食调理：**
    *   多吃益气固表的食物，如山药、黄芪（少量炖汤）、大枣、糯米等。
    *   避免生冷、辛辣刺激性食物，以免损伤脾胃，

In [15]:
import re

has_think = 0
all_sft = 0

for _ in tqdm(range(2048)):
    item = next(dd)

    tmp_text = tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100])

    pattern = r"<ggthink>(.*?)</ggthink>"
    matches = re.findall(pattern, tmp_text, re.DOTALL)

    if len(matches) == 0:
        continue

    all_sft += 1

    for i, match in enumerate(matches):
        if len(match.strip())>0:
            has_think += 1
            break


100%|██████████| 2048/2048 [01:06<00:00, 30.71it/s] 


In [29]:
has_think 

754

In [ ]:
(all_sft - has_think)

1107